## 네이버 블로그 크롤링

In [ ]:
# 라이브러리 불러오기
import pandas as pd # 데이터 조작 및 분석을 모듈
import time # 코드 실행 속도 조절을 위한 모듈
import re   # 정규표현식을 사용하여 텍스트에서 특수문자를 제거하거나 특정 패턴을 찾을 때 쓰는 모듈

from selenium import webdriver # 브라우저 자동화를 위한 모듈
from bs4 import BeautifulSoup as BS # HTML 내용 파싱을 위한 모듈
from selenium.webdriver.common.by import By # 다양한 방법으로 엘리먼트를 찾기 위한 모듈
from selenium.webdriver.support.ui import WebDriverWait # 브라우저가 요소를 찾을 때까지 대기(Wait)해주는 모듈
from selenium.webdriver.support import expected_conditions as EC # '어떤 상태'가 될 때까지 기다릴지 조건을 정해주는(EC) 기능 임포트

In [ ]:
# 브라우저 실행
driver = webdriver.Chrome()
# 브라우저 창 최대화 - 좋아요 수(공감 수)원활한 수집을 위해 : 최대화 안했을 때에 인식이 안됐던 점 보완
driver.maximize_window()

In [ ]:
import datetime # 날짜와 시간을 다루는 파이썬 기본 라이브러리

# 네이버 검색 결과의 무한 스크롤 특성으로 인해 처음 페이지에 접속하면 네이버는 서버 부하를 줄이기 위해 약 30개 정도의 글만 먼저 보여줌. 
# 따라서 사용자가 화면을 아래로 스크롤해야 추가로 글을 더 불러올 수 있음. -> 스코롤 다운 함수를 정의해 블로그 확보 갯수 조절 가능!

# 스크롤 다운 함수 정의
def doScrollDown(whileSeconds): 
    start = datetime.datetime.now() # 함수가 실행된 '현재 시점'의 시/분/초 가져오기
    end = start + datetime.timedelta(seconds=whileSeconds) # 입력받은 초 만큼의 시간 간격 생성
    while True:
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);') # 페이지 맨 아래로 스크롤 다운
        time.sleep(1.2) # 다음 목록이 로딩될 시간(약 1.2초)을 줌
        if datetime.datetime.now() > end: # 만약 현재 시간이 아까 계산해둔 종료 시간(end)보다 커지면 반복 break
            break 

In [ ]:
# (2026.02.28*미국 이란 전쟁 발발일*~2026.03.26) 동안 "미국 이란 전쟁"에 대한 네이버 블로그 검색 결과
naver_blog_url = 'https://search.naver.com/search.naver?ssc=tab.blog.all&query=%EB%AF%B8%EA%B5%AD%20%EC%9D%B4%EB%9E%80%20%EC%A0%84%EC%9F%81&sm=tab_opt&nso=so%3Ar%2Cp%3Afrom20260228to20260326'
driver.get(naver_blog_url)
time.sleep(2) # 검색 결과 로딩 대기

In [ ]:
# 2. 목록 확장을 위해 스크롤
doScrollDown(15)

In [ ]:
# [네이버 블로그 제목과 URL 추출 시작]

# 제목과 URL을 저장할 리스트 초기화 (나중에 카페와 구분하기 위해 blog_ 접두사 사용)
blog_title_list = []
blog_url_list = []

# 네이버가 최근 블로그 검색 결과 레이아웃을 업데이트하면서 클래스명이 무작위 문자열이 섞인 형태로 바뀐것을 F12키를 활용해 인지했습니다.
# 이런 클래스명은 네이버가 코드를 업데이트할 때마다 수시로 바뀌기 때문에, '클래스 이름' 대신 해당 클래스의 '구조'나 '속성'을 이용해 크롤링하는 것이 좋다고 느꼈습니다.
# <a> 태그 중에서 'data-heatmap-target' 속성값이 '.nblg'인 것만 골라내기
found_blog_elements = driver.find_elements(By.CSS_SELECTOR, 'a[data-heatmap-target=".nblg"]')

# 찾아낸 여러 개의 요소(found_blog_elements)를 하나씩 꺼내어 반복문 돌리기
for element in found_blog_elements:
    title_text = element.text.strip()
    url_link = element.get_attribute('href')
    
    if title_text and url_link: # 제목과 주소가 모두 정상적으로 존재할 때만 리스트에 추가 (빈 값 방지)
        blog_title_list.append(title_text) # 제목 리스트에 추가
        blog_url_list.append(url_link)     # 주소 리스트에 추가

# 중복 제거 (데이터 정제)
# 대량 수집 시 중복된 글을 피하기 위해 판다스 데이터프레임을 활용합니다.
df_blog_temp = pd.DataFrame({'title': blog_title_list, 'url': blog_url_list})

# subset=['url'] = 주소(url) 컬럼을 기준으로 중복된 블로그인지 검사해"라는 뜻
# keep='first' = 만약 똑같은 주소가 여러 개 발견되면, 가장 처음에 나온 것 하나만 남기고 나머지는 버려"라는 뜻
df_blog_temp = df_blog_temp.drop_duplicates(subset=['url'], keep='first')

# 중복 제거된 데이터를 다시 우리가 사용할 리스트로 변환
blog_title_list = df_blog_temp['title'].tolist()
blog_url_list = df_blog_temp['url'].tolist()

print(f"✅ 총 {len(blog_url_list)}개의 중복 없는 네이버 블로그 주소를 확보했습니다.")

✅ 총 420개의 중복 없는 네이버 블로그 주소를 확보했습니다.


In [ ]:
# --- [네이버 블로그 본문, 좋아요, 댓글, 이미지 및 영상 갯수 총 6개 항목 상세 수집 시작] ---
blog_new_doc = []      # 1. 본문
blog_like_cnt = []     # 2. 좋아요(공감) 수
blog_comment_cnt = []  # 3. 댓글 수
blog_comment_list = [] # 4. 댓글 내용
blog_img_cnt = []      # 5. 이미지 수
blog_div_cnt = []      # 6. 영상 수

# blog_url_list에 저장된 주소 개수만큼 반복문을 실행
for i in range(len(blog_url_list)): 
    driver.execute_script(f"window.open('{blog_url_list[i]}')")
    driver.switch_to.window(driver.window_handles[1]) # 새로 열린 탭(인덱스 1번)으로 제어권을 전환
    
    # 요소가 나타날 때까지만 최대 3초 대기하는 wait 설정
    wait = WebDriverWait(driver, 3) 
    
    try:
        # 네이버 블로그는 본문이 'mainFrame'이라는 iframe 안에 숨겨져 있어 전환이 필요
        # 프레임이 로딩되자마자 즉시 전환
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.ID, 'mainFrame')))
        
        html = driver.page_source
        soup = BS(html, 'html.parser')
        
        # 1. 본문 추출
        try:
            content = soup.select_one('.se-main-container, #postViewArea').get_text(strip=True)
        except: # 본문을 찾을 수 없는 경우
            content = "null"
        blog_new_doc.append(content)

        # 2. 좋아요(공감) 추출
        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") # 화면 아래에 있어야 공감 버튼 요소가 활성화되는 경우가 많아 바닥으로 스크롤
            
            # 좋아요 숫자가 보일 때까지만 딱 대기 (최대 2초)
            like_el = WebDriverWait(driver, 2).until(EC.presence_of_element_located((By.CSS_SELECTOR, "span.u_likeit_text._count.num")))
            like_val = like_el.get_attribute('textContent').strip() # .text 대신 'textContent'를 사용 -> 숨겨진 숫자까지 강제로 가져오기
            if not like_val or not like_val.isdigit(): like_val = "0" # 숫자가 비어있거나 숫자가 아니면 '0'으로 처리
        except:
            like_val = "0"
        blog_like_cnt.append(like_val)

        # 3. 댓글 수 추출
        try:
            comment_num = driver.find_element(By.ID, "commentCount").get_attribute('textContent').strip()
            if not comment_num: comment_num = "0"
        except:
            comment_num = "0"
        blog_comment_cnt.append(comment_num)

        # 4. 이미지/영상 수 추출
        # BeautifulSoup의 select 기능을 사용하여 이미지 태그와 영상 관련 클래스의 개수 세기
        blog_img_cnt.append(len(soup.select('img.se-image-resource')))
        blog_div_cnt.append(len(soup.select('.pzp-ui-dimmed, .se-video-container')))

        # 5. 댓글 내용 추출
        try:
            comment_btn = driver.find_element(By.ID, "commentCount") # 댓글 버튼을 찾아 자바스크립트로 강제 클릭
            driver.execute_script("arguments[0].click();", comment_btn) 
            time.sleep(1) 
            comments = driver.find_elements(By.CLASS_NAME, "u_cbox_contents") # 댓글 내용이 담긴 요소들 모두 찾기
            # 댓글 창은 정상적으로 열렸으나, 실제로 작성된 댓글이 0개인 경우
            blog_comment_final = "\n".join([c.text for c in comments]) if comments else "댓글 없음" 
        except:
            # 블로그 주인이 댓글 기능을 아예 막아 놓은 경우 (commentCount 요소를 찾지 못함)
            blog_comment_final = "댓글 막힘" 
        blog_comment_list.append(blog_comment_final)

    except Exception as e:
        print(f"⚠️ {i+1}번 글 오류: {e}")
        # 오류 시에도 리스트 길이를 맞추기 위해 기본값 삽입
        blog_new_doc.append("error")
        blog_like_cnt.append("0")
        blog_comment_cnt.append("0")
        blog_comment_list.append("error")
        blog_img_cnt.append(0)
        blog_div_cnt.append(0)

    # 작업이 끝난 탭을 닫고 다시 메인 목록 탭(인덱스 0번)으로 돌아가기
    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    
    # 잘 되고 있는지 10개마다 진행 상황 출력
    if (i + 1) % 10 == 0:
        print(f"[네이버 블로그] 진행 상황: {i+1}/{len(blog_url_list)} 완료")

driver.quit() # 브라우저 종료
print("수집 프로세스 종료!")

[네이버 블로그] 진행 상황: 10/420 완료
[네이버 블로그] 진행 상황: 20/420 완료
[네이버 블로그] 진행 상황: 30/420 완료
[네이버 블로그] 진행 상황: 40/420 완료
[네이버 블로그] 진행 상황: 50/420 완료
[네이버 블로그] 진행 상황: 60/420 완료
[네이버 블로그] 진행 상황: 70/420 완료
[네이버 블로그] 진행 상황: 80/420 완료
[네이버 블로그] 진행 상황: 90/420 완료
[네이버 블로그] 진행 상황: 100/420 완료
[네이버 블로그] 진행 상황: 110/420 완료
[네이버 블로그] 진행 상황: 120/420 완료
[네이버 블로그] 진행 상황: 130/420 완료
[네이버 블로그] 진행 상황: 140/420 완료
[네이버 블로그] 진행 상황: 150/420 완료
[네이버 블로그] 진행 상황: 160/420 완료
[네이버 블로그] 진행 상황: 170/420 완료
[네이버 블로그] 진행 상황: 180/420 완료
[네이버 블로그] 진행 상황: 190/420 완료
[네이버 블로그] 진행 상황: 200/420 완료
[네이버 블로그] 진행 상황: 210/420 완료
[네이버 블로그] 진행 상황: 220/420 완료
[네이버 블로그] 진행 상황: 230/420 완료
[네이버 블로그] 진행 상황: 240/420 완료
[네이버 블로그] 진행 상황: 250/420 완료
[네이버 블로그] 진행 상황: 260/420 완료
[네이버 블로그] 진행 상황: 270/420 완료
[네이버 블로그] 진행 상황: 280/420 완료
[네이버 블로그] 진행 상황: 290/420 완료
[네이버 블로그] 진행 상황: 300/420 완료
[네이버 블로그] 진행 상황: 310/420 완료
[네이버 블로그] 진행 상황: 320/420 완료
[네이버 블로그] 진행 상황: 330/420 완료
[네이버 블로그] 진행 상황: 340/420 완료
[네이버 블로그] 진행 상황: 350/420 완료
[네이버 블로그] 진행 상황: 360/420 완료
[

In [ ]:
# --- [네이버 블로그 최종 데이터프레임 생성 및 저장] ---

# 태하 님이 제안하신 구조를 바탕으로, 수집된 리스트들을 하나의 표로 합칩니다.
# 'ch'와 'ch2' 컬럼을 추가하여 나중에 카페 데이터와 섞여도 출처를 바로 알 수 있게 합니다.
df_blog = pd.DataFrame({
    'title': blog_title_list,      # 글 제목
    'url': blog_url_list,          # 글 주소 (원문 확인해야 할 상황 대비)
    'doc': blog_new_doc,           # 본문 내용
    'like': blog_like_cnt,         # 좋아요(공감) 수
    'comment_cnt': blog_comment_cnt, # 댓글 수
    'comment_list': blog_comment_list, # 댓글 내용
    'img': blog_img_cnt,           # 이미지 개수
    'div': blog_div_cnt,           # 영상 개수
    'ch': 'naver',                 # 채널 구분 1 (네이버/다음)
    'ch2': 'blog'                  # 채널 구분 2 (블로그/카페)
})

# 데이터 확인 (잘 들어갔는지 상위 5개 미리보기)
print("📊 네이버 블로그 데이터 생성 완료 (상위 5행):")
print(df_blog.head())

# 파일 저장 (파일명 규칙: naver_blog_data.csv)
file_name = 'naver_blog_data.csv'
df_blog.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"{file_name} 저장 완료! 이제 카페 수집으로 넘어갈 준비가 되었습니다.")

📊 네이버 블로그 데이터 생성 완료 (상위 5행):
                                        title  \
0           미국 이란 전쟁 5일간 휴전, 코스피 나스닥 불반등 시작될까   
1        미국-이란 전쟁 '유가 100달러' 시대… 에너지 취약계층 그림자   
2                 미국-이란 전쟁 확산에, 코스피 급락·방산주 폭등   
3  트럼프 미국 이란 전쟁 진짜 이유 뒤에 숨겨진 거대 자본의 소름 끼치는 설계   
4           미국 이란 전쟁 이유, 우리나라 경제 주가에 미치는 영향은?   

                                                 url  \
0    https://blog.naver.com/mellongi123/224227202490   
1  https://blog.naver.com/energyinfoplaza/2242178...   
2     https://blog.naver.com/newsystock/224202772150   
3        https://blog.naver.com/abcde10/224230745437   
4      https://blog.naver.com/kimgr1010/224220542560   

                                                 doc like comment_cnt  \
0  그야말로 도박장 같은 주식시장이다. 시퍼렇게 물들었던 계좌였는데 이제는 미국-이란 ...   43           5   
1  미국-이란 전쟁 '유가 100달러' 시대…에너지 취약계층 그림자세계는 다시 전쟁의 ...   38          19   
2  미국-이란 전쟁 확산에,코스피 급락·방산주 폭등​미국-이란 전쟁이 확장되며 호르무즈...   13           3   
3  ​​트럼프 미국 이란 전쟁 진짜 이유 뒤에 숨겨진 거대 자본의 소름 끼치는 설계

In [ ]:
# 브라우저 실행
driver = webdriver.Chrome()
# 브라우저 창 최대화 - 좋아요 수(공감 수)원활한 수집을 위해 : 최대화 안했을 때에 인식이 안됐던 점 보완
driver.maximize_window()

In [ ]:
# (2026.02.28*미국 이란 전쟁 발발일*~2026.03.26) 동안 "미국 이란 전쟁"에 대한 네이버 카페 검색 결과
# 네이버 카페 검색 옵션에서 '일반글' 선택으로 불필요한 데이터인 '거래글'수집 사전 차단.
naver_cafe_url = "https://search.naver.com/search.naver?cafe_where=articleg&nso=so%3Ar%2Cp%3Afrom20260228to20260326&nso_open=1&prdtype=0&query=%EB%AF%B8%EA%B5%AD+%EC%9D%B4%EB%9E%80+%EC%A0%84%EC%9F%81&sm=mtb_opt&ssc=tab.cafe.all&st=rel&stnm=rel&opt_tab=0&nso=so%3Ar%2Cp%3Afrom20260228to20260326&date_from=20260228&date_to=20260326"
driver.get(naver_cafe_url)
time.sleep(2) # 페이지 로딩 대기

In [ ]:
# 2. 목록 확장을 위해 스크롤
doScrollDown(15)

In [ ]:
# --- [네이버 카페 제목과 URL 추출 시작] ---

# 제목과 URL을 저장할 리스트 초기화 (나중에 블로그와 구분하기 위해 cafe_ 접두사 사용)
cafe_title_list = []
cafe_url_list = []

# BeautifulSoup으로 현재 페이지의 HTML 소스를 가져옵니다. (속도 최적화를 위해 BS 활용)
html = driver.page_source
soup = BS(html, 'html.parser')

# 네이버 카페 검색 결과의 게시글 요소들을 선택합니다.
# 카페 역시 클래스명이 바뀔 수 있으므로, 공통적인 구조인 '.view_wrap' 내부의 '.title_link'를 활용합니다.
found_cafe_elements = soup.select('.view_wrap .title_link')

# 찾아낸 여러 개의 요소들을 하나씩 꺼내어 반복문을 돌립니다.
for element in found_cafe_elements:
    title_text = element.get_text(strip=True)
    url_link = element['href']
    
    if title_text and url_link: # 제목과 주소가 모두 정상적으로 존재할 때만 리스트에 추가 (빈 값 방지)
        cafe_title_list.append(title_text) # 제목 리스트에 추가
        cafe_url_list.append(url_link)     # 주소 리스트에 추가

# [중복 제거 - 데이터 정제]
# 대량 수집 시 중복된 글을 피하기 위해 판다스 데이터프레임을 활용합니다.
df_cafe_temp = pd.DataFrame({'title': cafe_title_list, 'url': cafe_url_list})

# 주소(url) 컬럼을 기준으로 중복된 카페 글인지 검사하여 첫 번째 것만 남깁니다.
df_cafe_temp = df_cafe_temp.drop_duplicates(subset=['url'], keep='first')

# 중복 제거된 데이터를 다시 우리가 상세 수집에서 사용할 리스트로 변환
cafe_title_list = df_cafe_temp['title'].tolist()
cafe_url_list = df_cafe_temp['url'].tolist()

print(f"✅ 총 {len(cafe_url_list)}개의 중복 없는 네이버 카페 주소를 확보했습니다.")

✅ 총 420개의 중복 없는 네이버 카페 주소를 확보했습니다.


In [ ]:
# --- [네이버 카페 본문, 좋아요, 댓글, 이미지 및 영상 갯수 총 6개 항목 상세 수집 시작] ---

# 데이터를 저장할 리스트들 초기화 (네이버 카페용 접두사 cafe_ 사용)
cafe_new_doc = []      # 1. 본문
cafe_like_cnt = []     # 2. 좋아요(공감) 수
cafe_comment_cnt = []  # 3. 댓글 수
cafe_comment_list = [] # 4. 댓글 내용
cafe_img_cnt = []      # 5. 이미지 수
cafe_div_cnt = []      # 6. 영상 수

# cafe_url_list에 저장된 주소 개수만큼 반복문을 실행
# --- [네이버 카페 상세 수집] ---

for i in range(len(cafe_url_list)): 
    driver.execute_script(f"window.open('{cafe_url_list[i]}')")
    driver.switch_to.window(driver.window_handles[1])
    
    wait = WebDriverWait(driver, 3)
    
    try:
        # 1. 프레임 전환
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.ID, 'cafe_main')))
        time.sleep(0.5) # 안정적인 로딩을 위해 살짝 대기
        
        html = driver.page_source
        soup = BS(html, 'html.parser')
        
        # 1. 본문 추출
        try:
            content = soup.select_one('.se-main-container, .ContentRenderer, .article_viewer').get_text(strip=True)
        except: content = "내용 없음"
        cafe_new_doc.append(content)

        # 2. 좋아요 추출
        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            # a.like_count_btn 태그 안의 텍스트를 가져옵니다.
            like_el = WebDriverWait(driver, 2).until(EC.presence_of_element_located((By.CSS_SELECTOR, "em.u_cnt._count")))
            like_val = like_el.get_attribute('textContent').strip()
            if not like_val or not like_val.isdigit(): like_val = "0" # 숫자가 비어있거나 숫자가 아니면 '0'으로 처리
        except: like_val = "0"
        cafe_like_cnt.append(like_val)

        # 3. 댓글 수 추출
        try:
            # 댓글창 영역 근처의 strong.num을 찾습니다.
            comment_num_el = driver.find_element(By.CSS_SELECTOR, "strong.num, .comment_count")
            comment_num = comment_num_el.get_attribute('textContent').strip()
            comment_num = re.sub(r'[^0-9]', '', comment_num)
            if not comment_num: comment_num = "0"
        except: comment_num = "0"
        cafe_comment_cnt.append(comment_num)

        # 4. 이미지 및 영상 수 추출
        # 이미지: 클래스가 se-image-resource인 것들 중 .gif(움짤/영상)가 아닌 것만 필터링
        all_imgs = soup.select('img.se-image-resource')
        
        # 영상(움짤 포함): 확장자가 gif거나 특정 영상 클래스를 포함하는 경우
        # gif도 영상 수에 포함시키도록 설정
        video_list = [img for img in all_imgs if '.gif' in img.get('src', '').lower()]
        image_list = [img for img in all_imgs if '.gif' not in img.get('src', '').lower()]
        
        cafe_img_cnt.append(len(image_list))
        # 기존 영상 태그(.pzp-ui-dimmed 등)와 gif 리스트를 합산
        cafe_div_cnt.append(len(video_list) + len(soup.select('.pzp-ui-dimmed, .se-video-container')))

        # 5. 댓글 내용 추출
        try:
            comment_btn = driver.find_element(By.CSS_SELECTOR, "a.button_comment, .btn_comment")
            driver.execute_script("arguments[0].click();", comment_btn) 
            time.sleep(1) 
            comments = driver.find_elements(By.CLASS_NAME, "comment_text_view")
            cafe_comment_final = "\n".join([c.text for c in comments]) if comments else "댓글 없음"
        except:
            cafe_comment_final = "댓글 막힘" 
        cafe_comment_list.append(cafe_comment_final)

    except Exception as e:
        print(f"⚠️ {i+1}번 카페 글 오류: {e}")
        cafe_new_doc.append("에러 발생"); cafe_like_cnt.append("0"); cafe_comment_cnt.append("0")
        cafe_comment_list.append("에러 발생"); cafe_img_cnt.append(0); cafe_div_cnt.append(0)

    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    
    if (i + 1) % 10 == 0:
        print(f"[네이버 카페] 진행 상황: {i+1}/{len(cafe_url_list)} 완료")

driver.quit() # 브라우저 종료
print("수집 프로세스 종료!")

[네이버 카페] 진행 상황: 10/420 완료
[네이버 카페] 진행 상황: 20/420 완료
[네이버 카페] 진행 상황: 30/420 완료
[네이버 카페] 진행 상황: 40/420 완료
[네이버 카페] 진행 상황: 50/420 완료
[네이버 카페] 진행 상황: 60/420 완료
[네이버 카페] 진행 상황: 70/420 완료
[네이버 카페] 진행 상황: 80/420 완료
[네이버 카페] 진행 상황: 90/420 완료
[네이버 카페] 진행 상황: 100/420 완료
[네이버 카페] 진행 상황: 110/420 완료
[네이버 카페] 진행 상황: 120/420 완료
[네이버 카페] 진행 상황: 130/420 완료
[네이버 카페] 진행 상황: 140/420 완료
[네이버 카페] 진행 상황: 150/420 완료
[네이버 카페] 진행 상황: 160/420 완료
[네이버 카페] 진행 상황: 170/420 완료
[네이버 카페] 진행 상황: 180/420 완료
[네이버 카페] 진행 상황: 190/420 완료
[네이버 카페] 진행 상황: 200/420 완료
[네이버 카페] 진행 상황: 210/420 완료
[네이버 카페] 진행 상황: 220/420 완료
[네이버 카페] 진행 상황: 230/420 완료
[네이버 카페] 진행 상황: 240/420 완료
[네이버 카페] 진행 상황: 250/420 완료
[네이버 카페] 진행 상황: 260/420 완료
[네이버 카페] 진행 상황: 270/420 완료
[네이버 카페] 진행 상황: 280/420 완료
[네이버 카페] 진행 상황: 290/420 완료
[네이버 카페] 진행 상황: 300/420 완료
[네이버 카페] 진행 상황: 310/420 완료
[네이버 카페] 진행 상황: 320/420 완료
[네이버 카페] 진행 상황: 330/420 완료
[네이버 카페] 진행 상황: 340/420 완료
[네이버 카페] 진행 상황: 350/420 완료
[네이버 카페] 진행 상황: 360/420 완료
[네이버 카페] 진행 상황: 370/420 완료
[네이버 카페] 진

In [ ]:
# 상세 수집 단계에서 사용한 변수명과 동일하게 매칭
df_cafe = pd.DataFrame({
    'title': cafe_title_list,       # 카페 글 제목
    'url': cafe_url_list,           # 카페 글 주소
    'doc': cafe_new_doc,            # 카페 본문 내용
    'like': cafe_like_cnt,          # 좋아요 수 (F12로 찾은 like_count_btn 결과)
    'comment_cnt': cafe_comment_cnt, # 댓글 수 (F12로 찾은 strong.num 결과)
    'comment_list': cafe_comment_list, # 댓글 상세 내용 (댓글 막힘 포함)
    'img': cafe_img_cnt,            # 이미지 개수 (gif 제외)
    'div': cafe_div_cnt,            # 영상 및 움짤(gif) 개수
    'ch': 'naver',                  # 채널 구분 1
    'ch2': 'cafe'                   # 채널 구분 2
})

# 2. 데이터가 잘 들어갔는지 상위 5개 미리보기
print("📊 [검토] 네이버 카페 데이터프레임 상위 5행:")
print(df_cafe.head())

# 3. 엑셀에서 바로 열 수 있도록 CSV 파일로 저장
# 파일명: naver_cafe_data.csv
file_name = 'naver_cafe_data.csv'
df_cafe.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"\n💾 저장 완료! 파일명: {file_name}")
print(f"✅ 총 {len(df_cafe)}개의 카페 데이터가 정상적으로 기록되었습니다.")

📊 [검토] 네이버 카페 데이터프레임 상위 5행:
                  title                                                url  \
0    미국.이란 전쟁종식 시점 발표..  https://cafe.naver.com/ustock/5051217?art=ZXh0...   
1       미국-이란 전쟁타임라인 정리  https://cafe.naver.com/ralo24/186660?art=ZXh0Z...   
2       미국 이란 전쟁이유와 수혜주  https://cafe.naver.com/specup/7754714?art=ZXh0...   
3         미국 이란 전쟁예상도..  https://cafe.naver.com/likeusstock/1390157?art...   
4  미국-이란 전쟁요점만 정리해드립니다.  https://cafe.naver.com/dlxogns01/362930?art=ZX...   

                                                 doc like comment_cnt  \
0  게시판 안내를 확인해 주세요![필독] 평생주식카페는 6개월간 활동내용을 토대로 등급...   19          20   
1                                              내용 없음   22           0   
2  게시판 안내를 확인해 주세요!💭실시간 채용속보 받아보기 :https://vo.la/...    8          16   
3  게시판 안내를 확인해 주세요!👉우리 카페 활동에 필요한 주의사항 👈https://c...   19           6   
4  게시판 안내를 확인해 주세요!*** 자유게시판, 고민게시판에 투자 관련된 (주식, ...   77           0   

                                        comment_list  img  div  

In [ ]:
# 브라우저 실행
driver = webdriver.Chrome()
driver.maximize_window()

In [ ]:
# (2026.02.28*미국 이란 전쟁 발발일*~2026.03.26) 동안 "미국 이란 전쟁"에 대한 다음 카페 검색 결과
daum_cafe_url = 'https://search.daum.net/search?w=fusion&nil_search=btn&DA=STC&q=%EB%AF%B8%EA%B5%AD+%EC%9D%B4%EB%9E%80+%EC%A0%84%EC%9F%81&col=cafe&sd=20260228000000&ed=20260326235959&period=u'
driver.get(daum_cafe_url)
time.sleep(2) # 검색 결과 로딩 대기

In [ ]:
# 1. 리스트 초기화
daum_cafe_title_list = []  
daum_cafe_url_list = []     

# 검색어와 날짜 범위를 그대로 유지
base_url = "https://search.daum.net/search?w=fusion&nil_search=btn&DA=STC&q=%EB%AF%B8%EA%B5%AD+%EC%9D%B4%EB%9E%80+%EC%A0%84%EC%9F%81&col=cafe&sd=20260228000000&ed=20260326235959&period=u&p={}"

page_num = 1
target_count = 420  # 목표 개수 설정 :420

while len(daum_cafe_url_list) < target_count:
    # 해당 페이지로 강제 이동
    current_url = base_url.format(page_num)
    driver.get(current_url)
    time.sleep(2.5) # 다음 서버 부하 방지 및 로딩 대기
    
    # 현재 화면의 카드들 찾기
    cards = driver.find_elements(By.TAG_NAME, 'c-card')
    
    # 만약 페이지에 카드가 하나도 없다면 (검색 결과 끝) 종료
    if not cards:
        print(f"🏁 더 이상 검색 결과가 없습니다. (현재 수집량: {len(daum_cafe_url_list)}개)")
        break
        
    for card in cards:
        # 목표 개수 도달 시 즉시 중단
        if len(daum_cafe_url_list) >= target_count:
            break
            
        try:
            # 제목과 URL 추출 (c-title 태그 활용)
            title_el = card.find_element(By.TAG_NAME, 'c-title')
            title_text = title_el.text.strip()
            url_link = title_el.get_attribute('data-href')
            
            # 중복 체크 후 리스트에 담기
            if url_link not in daum_cafe_url_list:
                daum_cafe_title_list.append(title_text)
                daum_cafe_url_list.append(url_link)
        except:
            continue
    
    # 진행 상황 출력 (10단위로 끊어서 확인)
    if len(daum_cafe_url_list) % 10 == 0 or len(daum_cafe_url_list) == target_count:
        print(f"진행 상황: {len(daum_cafe_url_list)}/{target_count} 완료 (현재 {page_num}페이지)")

    # 다음 페이지 번호로 증가
    page_num += 1
    
    # 너무 무한 루프에 빠지지 않도록 최대 페이지 제한 (예: 50페이지)
    if page_num > 50:
        print("⚠️ 최대 페이지 제한(50p)에 도달하여 수집을 중단합니다.")
        break

# 2. 최종 데이터 정제 및 결과 출력
df_daum_list = pd.DataFrame({
    'title': daum_cafe_title_list,
    'url': daum_cafe_url_list
})

print(f"✅ 수집 프로세스 종료! 총 {len(df_daum_list)}개의 주소를 확보했습니다.")

진행 상황: 30/420 완료 (현재 3페이지)
진행 상황: 60/420 완료 (현재 5페이지)
진행 상황: 90/420 완료 (현재 7페이지)
진행 상황: 120/420 완료 (현재 9페이지)
진행 상황: 150/420 완료 (현재 11페이지)
🏁 더 이상 검색 결과가 없습니다. (현재 수집량: 157개)
✅ 수집 프로세스 종료! 총 157개의 주소를 확보했습니다.


In [ ]:
# 1. 데이터를 저장할 리스트들 초기화 (다음 카페용 접두사 daum_cafe_ 사용)
daum_cafe_doc_list = []
daum_cafe_like_cnt = []
daum_cafe_comment_cnt = []
daum_cafe_comment_list = []
daum_cafe_img_cnt = []
daum_cafe_video_cnt = []

for i, url in enumerate(daum_cafe_url_list):
    try:
        driver.get(url)
        time.sleep(3) # 페이지 로딩 대기
        
        # [프레임 전환]
        iframes = driver.find_elements(By.TAG_NAME, "iframe")
        found_frame = False
        for frame in iframes:
            f_name = frame.get_attribute('name') or frame.get_attribute('id')
            if f_name in ['down_frame', 'iframename']:
                driver.switch_to.frame(frame)
                found_frame = True
                break
        
        if not found_frame:
            try: driver.switch_to.frame(0); found_frame = True
            except: pass

        if found_frame:
            # 로딩을 위해 스크롤 다운
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.5)
            
            # 1. 본문 (그물망 전략)
            doc_text = "내용 없음"
            for selector in ['#article', '.tx-content-container', '.cont_view', 'body']:
                try:
                    target = driver.find_element(By.CSS_SELECTOR, selector)
                    if target.text.strip():
                        doc_text = target.text.strip().replace('\n', ' ')
                        break
                except: continue
            daum_cafe_doc_list.append(doc_text)

            # 2. 추천 수(좋아요 수)
            try:
                like_val = re.sub(r'[^0-9]', '', driver.find_element(By.CSS_SELECTOR, '.txt_item, .num_vote').text)
                if not like_val: like_val = "0"
            except: like_val = "0"
            daum_cafe_like_cnt.append(like_val)

            # 3. 댓글 내용 및 수
            try:
                comment_elements = driver.find_elements(By.CSS_SELECTOR, '.original_comment')
                c_list = [c.text.strip() for c in comment_elements if c.text.strip()]
                
                # 댓글 리스트의 개수를 숫자로 변환
                c_cnt = str(len(c_list))
                c_text = " | ".join(c_list) if c_list else "댓글 막힘"
            except:
                c_cnt = "0"
                c_text = "댓글 막힘"
            
            daum_cafe_comment_cnt.append(c_cnt)    # 매 루프마다 추가!
            daum_cafe_comment_list.append(c_text)  # 매 루프마다 추가!

            # 4. 이미지 및 영상 수
            i_cnt = len(driver.find_elements(By.CSS_SELECTOR, '.txc-image, img'))
            v_cnt = len(driver.find_elements(By.CSS_SELECTOR, '.cover_image_wrap, video, iframe[src*="youtube"]'))
            
            daum_cafe_img_cnt.append(i_cnt)
            daum_cafe_video_cnt.append(v_cnt)
        
        else:
            raise Exception("프레임 진입 실패")

        # 프레임 탈출
        driver.switch_to.default_content()

    except Exception as e:
        # 에러 발생 시에도 리스트 길이를 맞추기 위해 기본값 삽입 (네이버 코드 로직)
        print(f"⚠️ {i+1}번 다음 카페 글 오류: {e}")
        daum_cafe_doc_list.append("에러 발생")
        daum_cafe_like_cnt.append("0")
        daum_cafe_comment_cnt.append("0")
        daum_cafe_comment_list.append("댓글 막힘")
        daum_cafe_img_cnt.append(0)
        daum_cafe_video_cnt.append(0)
        driver.switch_to.default_content()

    if (i + 1) % 10 == 0:
        print(f"[다음 카페] 진행 상황: {i+1}/{len(daum_cafe_url_list)} 완료")

print("✅ 다음 카페 수집 완료!")

[다음 카페] 진행 상황: 10/157 완료
[다음 카페] 진행 상황: 20/157 완료
[다음 카페] 진행 상황: 30/157 완료
[다음 카페] 진행 상황: 40/157 완료
[다음 카페] 진행 상황: 50/157 완료
[다음 카페] 진행 상황: 60/157 완료
[다음 카페] 진행 상황: 70/157 완료
[다음 카페] 진행 상황: 80/157 완료
[다음 카페] 진행 상황: 90/157 완료
[다음 카페] 진행 상황: 100/157 완료
[다음 카페] 진행 상황: 110/157 완료
[다음 카페] 진행 상황: 120/157 완료
[다음 카페] 진행 상황: 130/157 완료
[다음 카페] 진행 상황: 140/157 완료
[다음 카페] 진행 상황: 150/157 완료
✅ 다음 카페 수집 완료!


In [ ]:
driver.quit() # 브라우저 종료

In [ ]:
# 1. 상세 수집 단계에서 사용한 다음 카페 변수명과 동일하게 매칭
df_daum_cafe = pd.DataFrame({
    'title': daum_cafe_title_list,       # 카페 글 제목
    'url': daum_cafe_url_list,           # 카페 글 주소
    'doc': daum_cafe_doc_list,           # 카페 본문 내용 (정밀 수집 결과)
    'like': daum_cafe_like_cnt,          # 추천 수 (.txt_item)
    'comment_cnt': daum_cafe_comment_cnt, # 댓글 수 (직접 카운팅 결과)
    'comment_list': daum_cafe_comment_list, # 댓글 상세 내용 (댓글 막힘 포함)
    'img': daum_cafe_img_cnt,             # 이미지 개수 (.txc-image)
    'div': daum_cafe_video_cnt,           # 영상 및 움짤 개수 (.cover_image_wrap)
    'ch': 'daum',                         # 채널 구분 1
    'ch2': 'cafe'                         # 채널 구분 2
})

# 2. 데이터가 잘 들어갔는지 상위 5개 미리보기
print("[검토] 다음 카페 데이터프레임 상위 5행:")
# 주요 데이터 위주로 먼저 확인합니다.
print(df_daum_cafe[['title', 'like', 'comment_cnt', 'div', 'doc']].head())

# 3. 엑셀에서 바로 열 수 있도록 CSV 파일로 저장
# 파일명: daum_cafe_data.csv
file_name = 'daum_cafe_data.csv'
df_daum_cafe.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"\n저장 완료! 파일명: {file_name}")
print(f"✅ 총 {len(df_daum_cafe)}개의 다음 카페 데이터가 정상적으로 기록되었습니다.")

[검토] 다음 카페 데이터프레임 상위 5행:
                           title like comment_cnt  div  \
0  "누가 물러나야 하나"…미국·이란 전쟁 끝낼 조건은?    0           0    0   
1  현재 트럼프의 오판으로 이상하게 됐다는 미국vs...    0          28    6   
2  미국-이란 전쟁 과연 어디로? 충격적인 이 내용...    3           2    0   
3          미국·이란 전쟁을 둘러싼 오해 [기고]    0           0    0   
4  미국 이란 전쟁.....쉽게 끝날지 걱정이 된다...    0           0    0   

                             doc  
0                          내용 없음  
1  출처: 여성시대 스피또띠아        전쟁 상...  
2  오늘 옛 트위터(현 X) 에 올라온 내용을 공유...  
3  미국·이란 전쟁을 둘러싼 오해 [기고] / 3월...  
4  기뢰는 설치는 쉽지만......제거하기는 어렵다...  

저장 완료! 파일명: daum_cafe_data.csv
✅ 총 157개의 다음 카페 데이터가 정상적으로 기록되었습니다.


In [ ]:
# 1. [전처리] 특수문자 제거 함수 정의
def clean_text(text):
    if pd.isna(text) or str(text).strip() == "": 
        return ""
    # 특수문자 제거
    cleaned = re.sub("[^0-9a-zA-Zㄱ-ㅎㅏ-ㅣ가-힣 ]", '', str(text))
    # 중복 공백 제거 및 양끝 공백 정리
    cleaned = ' '.join(cleaned.split())
    return cleaned

print(" 데이터 통합 및 전처리를 시작합니다 ")

try:
    # 2. 개별 수집 데이터 불러오기
    df_nb = pd.read_csv('naver_blog_data.csv')
    df_nc = pd.read_csv('naver_cafe_data.csv')
    df_dc = pd.read_csv('daum_cafe_data.csv')

    # 3. 데이터 통합 (Merge)
    raw_data = pd.concat([df_nb, df_nc, df_dc], ignore_index=True)
    initial_count = len(raw_data)

    # 4. [정제] 제목(title), 본문(doc), 댓글(comment_list) 모두 적용
    cols_to_clean = ['title', 'doc', 'comment_list']
    for col in cols_to_clean:
        if col in raw_data.columns:
            raw_data[col] = raw_data[col].apply(clean_text)
            # 정제 후 빈 값이 된 경우 '내용없음'으로 채움 (데이터 구조 유지)
            raw_data[col] = raw_data[col].replace('', '내용없음')

    # 5. [필터링] 본문(doc)이 2글자 미만인 행 삭제 (노이즈 제거)
    raw_data = raw_data[raw_data['doc'].str.len() >= 2]

    # 6. 인덱스 재설정 (삭제된 행 번호 정리)
    raw_data = raw_data.reset_index(drop=True)
    final_count = len(raw_data)

    # 7. 최종 파일 저장
    file_base_name = 'final_data_cleaned'
    
    # CSV 저장 (utf-8-sig로 엑셀 한글 깨짐 방지)
    raw_data.to_csv(f'{file_base_name}.csv', index=False, encoding='utf-8-sig')
    # Excel 저장
    raw_data.to_excel(f'{file_base_name}.xlsx', index=False)

    print("-" * 30)
    print(f"작업이 완료되었습니다!")
    print(f"📁 생성 파일: {file_base_name}.csv / {file_base_name}.xlsx")
    print("-" * 30)

except Exception as e:
    print(f"❌ 오류 발생: {e}")

 데이터 통합 및 전처리를 시작합니다 
------------------------------
작업이 완료되었습니다!
📁 생성 파일: final_data_cleaned.csv / final_data_cleaned.xlsx
------------------------------


In [ ]:
# 결측값 정리 (기사글의 내용이 비어있을 때 삭제 - 기사보다 블로그 글 등에서 걸러지는 경우 다수)

for i in range(len(docs)): # 본문
    if (len(docs['Document'][i])) < 2 or docs['Document'][i].isspace() == True:
        # 문서 내용이 두 글자 미만이나 공백 문서로만 돼 있는 경우 결측값 처리
        docs = docs.drop(i)

docs = docs.reset_index(drop=True) # 인덱스 재설정
docs

In [ ]:
# tqdm : 진행상황 상태를 사용자에게 바로 피드백해주는 라이브러리 -> 사용 권장!

# Kiwi(Korean Intelligent Word Identifier) : 최근 한국어 자연어 처리(NLP) 분야에서 주목받는 형태소 분석기
# 자바 기반인 KoNLPy의 한계를 극복하기 위해 C++로 작성

# !pip install kiwipiepy

# 토큰화 및 형태소 분석

### tqdm : 진행상황 상태를 사용자에게 바로 피드백해주는 라이브러리 -> 사용 권장!

### Kiwi(Korean Intelligent Word Identifier) : 최근 한국어 자연어 처리(NLP) 분야에서 주목받는 형태소 분석기
### 자바 기반인 KoNLPy의 한계를 극복하기 위해 C++로 작성

### !pip install kiwipiepy

In [ ]:
from kiwipiepy import Kiwi
from tqdm import tqdm

#1. 인스턴스(일꾼) 생성 - 반드시 소문자 변수에 담아서 사용
kiwi = Kiwi()

title_token_list = [] # 형태소 분석 전체 결과를 담을 리스트
title_token_noun = [] # 2글자 이상 명사만 담을 리스트

for i in tqdm(range(len(docs))):
    # 데이터가 비어있을 경우를 대비해 문자열로 변환
    text = str(docs['Title'][i])
    #2. 형태소 분석(Kiwi는 tokenize를 사용)
    tokens = kiwi.tokenize(text)

    # 전체 결과 저장: (단어, 품사) 튜플 형태로 변환하여 리스트에 추가
    pos_result = [(t.form, t.tag) for t in tokens]
    title_token_list.append(pos_result)

    #3. 명사 추출: 'NNG'(일반명사), 'NNP'(고유명사)이면서 길이가 2 이상인 것만 필터링
    nouns = [t.form for t in tokens if t.tag in ['NNG', 'NNP'] and len(t.form) > 1]
    title_token_noun.append(nouns)

print(title_token_noun)

: 

In [ ]:
Document_token_list = []
Document_token_noun = []

for i in tqdm(range(len(docs))):
    # 데이터가 비어있을 경우를 대비해 문자열로 변환
    text = str(docs['Document'][i])
    #2. 형태소 분석(Kiwi는 tokenize를 사용)
    tokens = kiwi.tokenize(text)

    # 전체 결과 저장: (단어, 품사) 튜플 형태로 변환하여 리스트에 추가
    pos_result = [(t.form, t.tag) for t in tokens]
    Document_token_list.append(pos_result)

    #3. 명사 추출: 'NNG'(일반명사), 'NNP'(고유명사)이면서 길이가 2 이상인 것만 필터링
    nouns = [t.form for t in tokens if t.tag in ['NNG', 'NNP'] and len(t.form) > 1]
    Document_token_noun.append(nouns)

print(Document_token_noun)
print(Document_token_list)

In [ ]:
import os
current_directory = os.getcwd()
print("현재 디렉토리:", current_directory)

In [ ]:
# 불용어 사전기반 불용어 리스트 정리
file_path = r'C:\Users\황태하\Crawling_practice'
f = open(file_path + '\stopwords-ko', "r", encoding = "UTF-8") # UTF-8로 저장된 한글 불용어 파일 읽어들여오기
st = f.readlines() # 불용어를 줄 단위 리스트로
f.close()
st[:10]

In [ ]:
stw = []
for i in range(0, len(st)):
    stw.append(st[i].rstrip('\n'))

stw

## 사용자의 필요 불용어를 불용어 사전에 추가 삽입

In [ ]:
# 검색 키워드에 대한 크롤링 데이터에서 불용어로 처리할 단어들을 지정
# '나만의 불용어 사전'
user_stopwords = []

stw.extend(user_stopwords)

In [ ]:
# 모든 불용어가 포함된 자신만의 사전을 csv 파일로 저장
import csv

with open('불용어2026.csv', 'w', encoding='utf-8-sig') as file : # 불용어에 특수문자 등 포함된 경우 읽어오기
    write = csv.writer(file)
    write.writerow(stw)

## 각 문서의 제목과 본문, 댓글내용에서 불용어 제거

In [ ]:
for word in stw:
    for i in range(0, len(title_token_noun)):
        # 리스트에 불용어가 있을 경우 제거
        while word in title_token_noun[i]:
            title_token_noun[i].remove(word)

for word in stw:
    for i in range(0, len(Document_token_noun)):
        while word in Document_token_noun[i]:
            Document_token_noun[i].remove(word)

# 문서파일 docs로 적용하여 각각의 불용어 제거
docs['title_token_list_pos'] = title_token_list # 형태소와 품사 리스트
docs['title_token_noun'] = title_token_noun     # 명사 리스트
docs['Document_token_noun'] = Document_token_noun
docs['Document_token_list_pos'] = Document_token_list

## 불용어 처리 후 최종 파일 저장 및 불러오기

In [ ]:
import pickle
f = open(file_path + 'total_estate_press.pkl', 'wb') # total_doc.pkl로 저장
pickle.dump(docs, f)
f.close()

In [ ]:
f = open(file_path + "total_estate_press.pkl", "wb") # total_doc.pkl로 저장
data = pickle.load(f)
f.close()

data